# Campagne CPU LUT uniquement

Notebook autonome (independant) pour executer uniquement la methode `cpu_lut_angles`.

In [1]:
# ==========================================================
# CELLULE 1 - CONFIGURATION (AUTONOME)
# ==========================================================
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
    HAS_DISPLAY = True
except Exception:
    HAS_DISPLAY = False

try:
    from ipywidgets import Dropdown, IntText, SelectMultiple, VBox, Label, Output
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

FS_DEFAULT = 11_999_000.0
IF_DEFAULT = 3_563_000.0
N_DEFAULT = 11999
NB_PHASES_DEFAULT = 1023
EPS = 1e-12

DDS_LUT_SIZE = 1024
DDS_PHASE_BITS = 32
DDS_LUT_BITS = 10

HEADER_RE = re.compile(
    r"PRN=(?P<prn>-?\d+)"
    r".*SNR=(?P<snr>-?\d+(?:\.\d+)?)dB"
    r".*Doppler=(?P<doppler>-?\d+(?:\.\d+)?)Hz"
    r".*CA_SHIFT=(?P<ca>-?\d+)chips",
    re.IGNORECASE,
)

METHOD_LABELS = {
    "cpu_lut_angles": "CPU LUT angles",
    "fpga": "FPGA",
}

EXPORT_DIR_DEFAULT = "Resultats validation SA"

def load_dds_luts_from_header(header_file_path):
    with open(header_file_path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()

    sin_match = re.search(r"static const osc_t DDS_SIN_LUT\[DDS_LUT_SIZE\] = \{(.*?)\};", content, re.DOTALL)
    if sin_match:
        sin_values = re.findall(r"\(osc_t\)([0-9e\.\-\+]+)", sin_match.group(1))
        dds_sin_lut = np.array([float(v) for v in sin_values], dtype=np.float32)
    else:
        raise ValueError("DDS_SIN_LUT not found in header file")

    cos_match = re.search(r"static const osc_t DDS_COS_LUT\[DDS_LUT_SIZE\] = \{(.*?)\};", content, re.DOTALL)
    if cos_match:
        cos_values = re.findall(r"\(osc_t\)([0-9e\.\-\+]+)", cos_match.group(1))
        dds_cos_lut = np.array([float(v) for v in cos_values], dtype=np.float32)
    else:
        raise ValueError("DDS_COS_LUT not found in header file")

    return dds_sin_lut, dds_cos_lut

DDS_SIN_LUT, DDS_COS_LUT = load_dds_luts_from_header("dds_lut_rom.h")


def _methods_export_tag(cfg):
    methods = [str(m) for m in cfg.get("run_methods", []) if str(m)]
    if not methods:
        return "no_method"
    return "_vs_".join(methods)


def resolve_export_dir(cfg):
    base = Path(str(cfg.get("export_dir", EXPORT_DIR_DEFAULT)))
    if bool(cfg.get("export_methods_subdir", True)):
        return base / _methods_export_tag(cfg)
    return base


DEFAULT_CONFIG = {
    "signals_dir": "signals",
    "prn_dir": "PRN",
    "signal_glob": "*.csv",
    "limit_signals": None,
    "run_methods": ["cpu_lut_angles"],
    "prn_list": None,
    "fs_hz": FS_DEFAULT,
    "if_hz": IF_DEFAULT,
    "n_samples": N_DEFAULT,
    "nb_phases": NB_PHASES_DEFAULT,
    "fd_start": -10000,
    "fd_end": 10000,
    "fd_step": 500,
    "detection_ratio_min": 2.5,
    "winner_margin_db_min": 3.0,
    "doppler_tol_hz": 500,
    "phase_tol_chip": 3,
    "bit_path": "sadfsnr.bit",
    "prn_mode": "all_available",
    "prn_search_mode": "injected_plus_decoys",
    "pfa_decoy_count": 8,
    "pfa_decoy_seed": 42,
    "resume_from_partial": False,
    "export_prefix": "validation_cpu_lut",
    "export_dir": "Resultats validation SA/cpu_lut_only",
    "export_methods_subdir": False,
    "autosave_every_acq": 10,
}


def collect_signal_files_for_ui(base_dir):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        return []
    files = sorted(base_dir.glob("*.csv"))
    valid_files = []
    for file_path in files:
        try:
            with file_path.open("r", encoding="utf-8", errors="ignore") as handle:
                first = handle.readline().strip()
            if HEADER_RE.search(first):
                valid_files.append(file_path.name)
        except Exception:
            continue
    return valid_files


def get_available_prns_for_ui(base_dir):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        return []
    prns = []
    patterns = ["PRN-*.csv", "prn-*.csv", "PRN_*.csv", "prn_*.csv"]
    for pattern in patterns:
        for file_path in base_dir.glob(pattern):
            match = re.search(r"(\d+)", file_path.stem)
            if match:
                prns.append(int(match.group(1)))
    return sorted(set(prns))


WIDGET_STATE = {
    "enabled": False,
    "limit_mode": None,
    "limit_value": None,
    "fd_mode": None,
    "fd_start": None,
    "fd_end": None,
    "fd_step": None,
    "prn_mode": None,
    "prn_selector": None,
    "prn_search_mode": None,
}


def get_current_config():
    cfg = dict(DEFAULT_CONFIG)
    if not WIDGET_STATE["enabled"]:
        return cfg

    cfg["limit_signals"] = None if int(WIDGET_STATE["limit_mode"].value) == 0 else int(WIDGET_STATE["limit_value"].value)

    if int(WIDGET_STATE["fd_mode"].value) == 0:
        cfg["fd_start"] = -10000
        cfg["fd_end"] = 10000
        cfg["fd_step"] = 500
    else:
        cfg["fd_start"] = int(WIDGET_STATE["fd_start"].value)
        cfg["fd_end"] = int(WIDGET_STATE["fd_end"].value)
        cfg["fd_step"] = int(WIDGET_STATE["fd_step"].value)

    if int(WIDGET_STATE["prn_mode"].value) == 0:
        cfg["prn_list"] = None
    else:
        cfg["prn_list"] = list(WIDGET_STATE["prn_selector"].value)

    psm = int(WIDGET_STATE["prn_search_mode"].value)
    cfg["prn_search_mode"] = "injected_only" if psm == 0 else ("injected_plus_decoys" if psm == 1 else "full_grid")
    return cfg


_available_signals = collect_signal_files_for_ui(DEFAULT_CONFIG["signals_dir"])
_available_prns = get_available_prns_for_ui(DEFAULT_CONFIG["prn_dir"])

print("=== CONFIGURATION INTERACTIVE ===")
print(f"Signaux trouves: {len(_available_signals)}")
print(f"PRN disponibles: {_available_prns}")

if HAS_WIDGETS and HAS_DISPLAY and _available_prns:
    limit_mode = Dropdown(options=[("Tous", 0), ("Limiter", 1)], value=0, description="Signaux")
    limit_value = IntText(value=min(10, max(1, len(_available_signals))), description="Nb")
    fd_mode = Dropdown(options=[("Standard", 0), ("Personnalise", 1)], value=0, description="Doppler")
    fd_start = IntText(value=DEFAULT_CONFIG["fd_start"], description="Fd debut")
    fd_end = IntText(value=DEFAULT_CONFIG["fd_end"], description="Fd fin")
    fd_step = IntText(value=DEFAULT_CONFIG["fd_step"], description="Fd step")
    prn_mode = Dropdown(options=[("Tous PRN", 0), ("Selection", 1)], value=0, description="PRN")
    prn_selector = SelectMultiple(options=[(str(p), p) for p in _available_prns], value=tuple(_available_prns[:1]), rows=min(10, len(_available_prns)), description="Liste")
    prn_search_mode = Dropdown(
        options=[("Injecte seulement", 0), ("Injecte + leurres", 1), ("Grille complete", 2)],
        value=1,
        description="Recherche",
    )
    config_output = Output()

    def _refresh(_=None):
        cfg = get_current_config()
        with config_output:
            config_output.clear_output()
            print("=== CONFIG EFFECTIVE (selon menu) ===")
            print(pd.Series(cfg, dtype="object"))

    def _on_limit_value_change(change):
        if int(limit_mode.value) == 0 and change.get("new") is not None:
            limit_mode.value = 1

    def _on_prn_selector_change(change):
        if int(prn_mode.value) == 0 and change.get("new"):
            prn_mode.value = 1

    def _on_fd_custom_change(change):
        if int(fd_mode.value) == 0:
            fd_mode.value = 1

    for w in [limit_mode, limit_value, fd_mode, fd_start, fd_end, fd_step, prn_mode, prn_selector, prn_search_mode]:
        w.observe(_refresh, names="value")

    limit_value.observe(_on_limit_value_change, names="value")
    prn_selector.observe(_on_prn_selector_change, names="value")
    fd_start.observe(_on_fd_custom_change, names="value")
    fd_end.observe(_on_fd_custom_change, names="value")
    fd_step.observe(_on_fd_custom_change, names="value")

    WIDGET_STATE.update({
        "enabled": True,
        "limit_mode": limit_mode,
        "limit_value": limit_value,
        "fd_mode": fd_mode,
        "fd_start": fd_start,
        "fd_end": fd_end,
        "fd_step": fd_step,
        "prn_mode": prn_mode,
        "prn_selector": prn_selector,
        "prn_search_mode": prn_search_mode,
    })

    display(VBox([
        Label("Choix de campagne CPU LUT"),
        limit_mode, limit_value,
        fd_mode, fd_start, fd_end, fd_step,
        prn_mode, prn_selector,
        prn_search_mode,
        config_output,
    ]))
    _refresh()
else:
    print("[INFO] Widgets non disponibles ou PRN introuvables: configuration par defaut utilisee.")

CONFIG = get_current_config()
print()
print("[INFO] La configuration effective se met a jour dans le panneau interactif ci-dessus.")

=== CONFIGURATION INTERACTIVE ===
Signaux trouves: 86
PRN disponibles: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]



[INFO] La configuration effective se met a jour dans le panneau interactif ci-dessus.


In [2]:
# ==========================================================
# CELLULE 2 - CAMPAGNE DE VALIDATION
# ==========================================================
FPGA_CONTEXT = None


def to_s32(x):
    x = int(x)
    return x if x < 0x80000000 else x - 0x100000000


def phase_error_mod(measured, expected, modulo=1023):
    d = (int(measured) - int(expected)) % int(modulo)
    return int(min(d, modulo - d))


def parse_meta_from_header(csv_file):
    csv_file = Path(csv_file)
    with csv_file.open("r", encoding="utf-8", errors="ignore") as handle:
        first = handle.readline().strip()
    match = HEADER_RE.search(first)
    if not match:
        raise ValueError(f"Metadonnees introuvables dans {csv_file.name}")
    return {
        "prn": int(match.group("prn")),
        "snr": float(match.group("snr")),
        "doppler": int(round(float(match.group("doppler")))),
        "ca_shift": int(match.group("ca")),
    }


def load_signal_pm1(csv_file):
    csv_file = Path(csv_file)
    with csv_file.open("r", encoding="utf-8", errors="ignore") as handle:
        line1 = handle.readline().strip()
        line2 = handle.readline().strip()
    if line1 and not line1.startswith("#") and not line2:
        line2 = line1
    if not line2:
        raise ValueError(f"Signal vide: {csv_file}")
    data = np.fromstring(line2, sep=",", dtype=np.float32)
    if data.size == 0:
        raise ValueError(f"Format invalide: {csv_file}")
    data = np.where(data == 255, -1, data).astype(np.float32)
    return data


def load_prn_sequence(prn_dir, prn, n_samples):
    prn_dir = Path(prn_dir)
    candidates = [
        prn_dir / f"PRN-{prn}.csv",
        prn_dir / f"prn-{prn}.csv",
        prn_dir / f"PRN_{prn}.csv",
        prn_dir / f"prn_{prn}.csv",
    ]
    prn_path = next((path for path in candidates if path.exists()), None)
    if prn_path is None:
        raise FileNotFoundError(f"PRN {prn} introuvable dans {prn_dir}")
    seq = np.genfromtxt(prn_path, delimiter=",", dtype=np.float64)
    seq = np.atleast_1d(seq).reshape(-1)
    if seq.size < n_samples:
        raise ValueError(f"PRN trop court ({seq.size}) dans {prn_path}, attendu >= {n_samples}")
    return seq[:n_samples].astype(np.float32)


def get_available_prns(prn_dir):
    prn_dir = Path(prn_dir)
    prns = []
    patterns = ["PRN-*.csv", "prn-*.csv", "PRN_*.csv", "prn_*.csv"]
    for pattern in patterns:
        for file_path in prn_dir.glob(pattern):
            match = re.search(r"(\d+)", file_path.stem)
            if match:
                prns.append(int(match.group(1)))
    return sorted(set(prns))


def collect_signal_files(signals_dir, signal_glob):
    signals_dir = Path(signals_dir)
    files = sorted(signals_dir.glob(signal_glob)) if signals_dir.exists() else []
    if not files and signals_dir.exists():
        files = sorted(signals_dir.glob("*.csv"))
    valid_files = []
    for file_path in files:
        try:
            parse_meta_from_header(file_path)
            valid_files.append(file_path)
        except Exception:
            continue
    return valid_files


def _precompute_prn_shifts(prn, n_samples, nb_phases):
    shifts = np.floor(np.arange(nb_phases) * n_samples / nb_phases).astype(int)
    prn_shifted = np.empty((nb_phases, n_samples), dtype=np.float32)
    for tau in range(nb_phases):
        prn_shifted[tau] = np.roll(prn, -shifts[tau])
    return prn_shifted


def cpu_lut_angles_acquisition(signal, prn, cfg):
    t_total0 = time.perf_counter()

    n_samples = int(cfg["n_samples"])
    nb_phases = int(cfg["nb_phases"])
    fs_hz = float(cfg["fs_hz"])
    if_hz = float(cfg["if_hz"])
    fd_start = int(cfg["fd_start"])
    fd_end = int(cfg["fd_end"])
    fd_step = int(cfg["fd_step"])
    ratio_min = float(cfg["detection_ratio_min"])

    if fd_step <= 0:
        fd_step = 1

    signal = np.asarray(signal[:n_samples], dtype=np.complex64)
    prn = np.asarray(prn[:n_samples], dtype=np.float32)
    prep_done = time.perf_counter()

    prn_shifted = _precompute_prn_shifts(prn, n_samples, nb_phases)

    if fd_start <= fd_end:
        doppler_range = np.arange(fd_start, fd_end + 1, fd_step, dtype=np.int32)
    else:
        doppler_range = np.arange(fd_start, fd_end - 1, -fd_step, dtype=np.int32)
    corr_matrix = np.zeros((len(doppler_range), nb_phases), dtype=np.float64)

    t_kernel0 = time.perf_counter()
    n_idx = np.arange(n_samples, dtype=np.uint64)
    shift = DDS_PHASE_BITS - DDS_LUT_BITS

    for k, fd in enumerate(doppler_range):
        scale = 1 << DDS_PHASE_BITS
        num = int(-(if_hz + float(fd)) * scale)
        den = int(fs_hz)
        if num >= 0:
            num += den // 2
        else:
            num -= den // 2
        phase_inc = int(num // den) & 0xFFFFFFFF

        phase_acc = (n_idx * np.uint64(phase_inc)) & np.uint64(0xFFFFFFFF)
        lut_idx = ((phase_acc >> np.uint64(shift)) & np.uint64(DDS_LUT_SIZE - 1)).astype(np.int64)

        c = DDS_COS_LUT[lut_idx].astype(np.float64, copy=False)
        s = DDS_SIN_LUT[lut_idx].astype(np.float64, copy=False)
        data_mix = signal.astype(np.complex128, copy=False) * (c - 1j * s)

        corr = prn_shifted @ data_mix
        corr_matrix[k, :] = np.abs(corr) ** 2

    t_kernel1 = time.perf_counter()

    row, col = np.unravel_index(int(np.argmax(corr_matrix)), corr_matrix.shape)
    peak = float(corr_matrix[row, col])
    mean_corr = float(np.mean(corr_matrix))
    peak_ratio = peak / mean_corr if mean_corr > EPS else 0.0
    t_total1 = time.perf_counter()

    return {
        "method": "cpu_lut_angles",
        "detected": int(peak_ratio >= ratio_min),
        "doppler": int(doppler_range[row]),
        "phase": int(col),
        "peak": peak,
        "peak_ratio": peak_ratio,
        "time_s": float(t_kernel1 - t_kernel0),
        "time_kernel_s": float(t_kernel1 - t_kernel0),
        "time_prepare_s": float(prep_done - t_total0),
        "time_dma_s": 0.0,
        "time_ip_wait_s": 0.0,
        "time_end_to_end_s": float(t_total1 - t_total0),
    }


def run_method(method_name, signal, prn_seq, cfg):
    if method_name == "cpu_lut_angles":
        return cpu_lut_angles_acquisition(signal, prn_seq, cfg)
    raise ValueError(f"Methode inconnue dans ce notebook: {method_name}")


def build_winner_table(df_raw, cfg):
    winner_rows = []
    for (file_name, method_name), group in df_raw.groupby(["file", "method"], dropna=False):
        valid_group = group[group["status"] == "ok"].sort_values("peak_ratio", ascending=False).reset_index(drop=True)
        if valid_group.empty:
            error_row = group.iloc[0]
            winner_rows.append({
                "file": file_name,
                "method": method_name,
                "status": error_row["status"],
                "error_message": error_row.get("error_message", ""),
            })
            continue

        top1 = valid_group.iloc[0]
        top2_ratio = float(valid_group.iloc[1]["peak_ratio"]) if len(valid_group) > 1 else np.nan
        winner_margin_db = float(10.0 * np.log10((float(top1["peak_ratio"]) + EPS) / (top2_ratio + EPS))) if not np.isnan(top2_ratio) else np.nan
        robust_detected = int((float(top1["peak_ratio"]) >= float(cfg["detection_ratio_min"])) and (np.isnan(winner_margin_db) or winner_margin_db >= float(cfg["winner_margin_db_min"])))

        doppler_err = int(abs(int(top1["doppler_meas_hz"]) - int(top1["doppler_true_hz"])))
        phase_err = int(phase_error_mod(int(top1["phase_meas_chip"]), int(top1["phase_true_chip"]), int(cfg["nb_phases"])))
        prn_ok = int(int(top1["prn_tested"]) == int(top1["prn_injected"]))
        doppler_ok = int(doppler_err <= int(cfg["doppler_tol_hz"]))
        phase_ok = int(phase_err <= int(cfg["phase_tol_chip"]))
        strict_success = int((robust_detected == 1) and (prn_ok == 1) and (doppler_ok == 1) and (phase_ok == 1))
        false_alarm = int((robust_detected == 1) and (prn_ok == 0))
        miss = int(robust_detected == 0)

        winner_rows.append({
            "file": file_name,
            "method": method_name,
            "status": "ok",
            "error_message": "",
            "snr_db": float(top1["snr_db"]),
            "prn_injected": int(top1["prn_injected"]),
            "prn_winner": int(top1["prn_tested"]),
            "doppler_true_hz": int(top1["doppler_true_hz"]),
            "doppler_meas_hz": int(top1["doppler_meas_hz"]),
            "doppler_err_hz": doppler_err,
            "phase_true_chip": int(top1["phase_true_chip"]),
            "phase_meas_chip": int(top1["phase_meas_chip"]),
            "phase_err_chip": phase_err,
            "peak": float(top1["peak"]),
            "peak_ratio": float(top1["peak_ratio"]),
            "second_peak_ratio": top2_ratio,
            "winner_margin_db": winner_margin_db,
            "time_s": float(top1["time_s"]),
            "classic_detected": int(top1["detected"]),
            "robust_detected": robust_detected,
            "pd_flag": robust_detected,
            "pfa_flag": false_alarm,
            "pm_flag": miss,
            "prn_ok": prn_ok,
            "doppler_ok": doppler_ok,
            "phase_ok": phase_ok,
            "strict_success": strict_success,
        })
    return pd.DataFrame(winner_rows)


cfg = get_current_config()
CONFIG = dict(cfg)

signals_dir = Path(cfg["signals_dir"])
prn_dir = Path(cfg["prn_dir"])

signal_files = collect_signal_files(signals_dir, cfg["signal_glob"])
if cfg.get("limit_signals"):
    signal_files = signal_files[: int(cfg["limit_signals"])]

available_prns = get_available_prns(prn_dir)
selected_prns = [int(prn) for prn in cfg["prn_list"] if int(prn) in available_prns] if cfg.get("prn_list") else available_prns

_psm = str(cfg.get("prn_search_mode", "full_grid"))
if _psm == "injected_only":
    _n_prn_per_sig = 1
elif _psm == "injected_plus_decoys":
    _nd = int(cfg.get("pfa_decoy_count", 8))
    _n_prn_per_sig = 1 + min(_nd, max(0, len(selected_prns) - 1))
else:
    _n_prn_per_sig = len(selected_prns)

_n_acq = len(signal_files) * _n_prn_per_sig * len(cfg["run_methods"])
print(f"Acquisitions prevues: {_n_acq}")
print(f"Methodes: {cfg['run_methods']}")

_export_pre = resolve_export_dir(cfg)
_export_pre.mkdir(parents=True, exist_ok=True)
_prefix_pre = str(cfg["export_prefix"])
_raw_prev = _export_pre / f"{_prefix_pre}_raw_results.csv"
_raw_partial = _export_pre / f"{_prefix_pre}_raw_partial.csv"

_done_triples = set()
_df_prev = None
if bool(cfg.get("resume_from_partial")):
    _resume_src = _raw_prev if _raw_prev.exists() else (_raw_partial if _raw_partial.exists() else None)
    if _resume_src is not None:
        _df_prev = pd.read_csv(_resume_src)
        for _, _r in _df_prev.iterrows():
            _done_triples.add((str(_r["file"]), int(_r["prn_tested"]), str(_r["method"])))

new_rows = []
prn_cache = {}
_total_acq_planned = max(1, int(_n_acq))
_acq_counter = 0
_autosave_every = max(1, int(cfg.get("autosave_every_acq", 10)))


def _flush_partial_rows():
    if not new_rows:
        return
    _df_new = pd.DataFrame(new_rows)
    _df_tmp = pd.concat([_df_prev, _df_new], ignore_index=True) if (_df_prev is not None and not _df_prev.empty) else _df_new
    _df_tmp.to_csv(_raw_partial, index=False)


for index_signal, sig_file in enumerate(signal_files, 1):
    meta = parse_meta_from_header(sig_file)
    prn_injected = int(meta["prn"])
    doppler_true_hz = int(meta["doppler"])
    phase_true_chip = int(meta["ca_shift"] % int(cfg["nb_phases"]))
    snr_db = float(meta["snr"])

    signal = load_signal_pm1(sig_file)
    n_samples = int(cfg["n_samples"])
    signal = np.pad(signal, (0, max(0, n_samples - signal.size)), mode="edge")[:n_samples].astype(np.complex64)

    _mode_prn = str(cfg.get("prn_search_mode", "full_grid"))
    if _mode_prn == "injected_only":
        prns_this_signal = [prn_injected]
    elif _mode_prn == "injected_plus_decoys":
        _n_decoy = int(cfg.get("pfa_decoy_count", 8))
        _seed = int(cfg.get("pfa_decoy_seed", 42))
        _others = [int(p) for p in selected_prns if int(p) != int(prn_injected)]
        if not _others:
            prns_this_signal = [prn_injected]
        else:
            _k = min(_n_decoy, len(_others))
            _rng = np.random.default_rng(_seed + index_signal)
            _pick = _rng.choice(np.array(_others, dtype=np.int64), size=_k, replace=False)
            prns_this_signal = [int(prn_injected)] + sorted(int(x) for x in _pick.tolist())
    else:
        prns_this_signal = selected_prns

    for prn_tested in prns_this_signal:
        prn_global_t0 = time.perf_counter()
        prn_acq_count = 0
        if prn_tested not in prn_cache:
            prn_cache[prn_tested] = load_prn_sequence(prn_dir, prn_tested, n_samples)
        prn_seq = prn_cache[prn_tested]

        for method_name in cfg["run_methods"]:
            if (sig_file.name, int(prn_tested), str(method_name)) in _done_triples:
                _acq_counter += 1
                prn_acq_count += 1
                continue
            try:
                t_e2e0 = time.perf_counter()
                result = run_method(method_name, signal, prn_seq, cfg)
                t_e2e1 = time.perf_counter()
                end_to_end_s = float(t_e2e1 - t_e2e0)
                _acq_counter += 1
                prn_acq_count += 1
                print(
                    f"[{_acq_counter}/{_total_acq_planned}] {method_name} PRN={prn_tested} SNR={snr_db:g} "
                    f"DONE global={end_to_end_s:.3f}s"
                )
                new_rows.append({
                    "file": sig_file.name,
                    "method": method_name,
                    "status": "ok",
                    "error_message": "",
                    "snr_db": snr_db,
                    "prn_injected": prn_injected,
                    "prn_tested": int(prn_tested),
                    "doppler_true_hz": doppler_true_hz,
                    "phase_true_chip": phase_true_chip,
                    "detected": int(result["detected"]),
                    "doppler_meas_hz": int(result["doppler"]),
                    "phase_meas_chip": int(result["phase"]),
                    "peak": float(result["peak"]),
                    "peak_ratio": float(result["peak_ratio"]),
                    "time_s": float(result["time_s"]),
                    "time_kernel_s": float(result.get("time_kernel_s", result["time_s"])),
                    "time_prepare_s": float(result.get("time_prepare_s", np.nan)),
                    "time_dma_s": float(result.get("time_dma_s", 0.0)),
                    "time_ip_wait_s": float(result.get("time_ip_wait_s", 0.0)),
                    "time_end_to_end_s": end_to_end_s,
                })
                if (_acq_counter % _autosave_every) == 0:
                    _flush_partial_rows()
            except Exception as exc:
                _acq_counter += 1
                prn_acq_count += 1
                print(f"[{_acq_counter}/{_total_acq_planned}] {method_name} PRN={prn_tested} SNR={snr_db:g} ERROR: {exc}")
                new_rows.append({
                    "file": sig_file.name,
                    "method": method_name,
                    "status": "error",
                    "error_message": str(exc),
                    "snr_db": snr_db,
                    "prn_injected": prn_injected,
                    "prn_tested": int(prn_tested),
                    "doppler_true_hz": doppler_true_hz,
                    "phase_true_chip": phase_true_chip,
                    "detected": 0,
                    "doppler_meas_hz": np.nan,
                    "phase_meas_chip": np.nan,
                    "peak": np.nan,
                    "peak_ratio": np.nan,
                    "time_s": np.nan,
                    "time_kernel_s": np.nan,
                    "time_prepare_s": np.nan,
                    "time_dma_s": np.nan,
                    "time_ip_wait_s": np.nan,
                    "time_end_to_end_s": np.nan,
                })
                if (_acq_counter % _autosave_every) == 0:
                    _flush_partial_rows()

        prn_global_elapsed = time.perf_counter() - prn_global_t0
        print(
            f"[PRN-TOTAL] file={sig_file.name} PRN={prn_tested} acquisitions={prn_acq_count} "
            f"global_elapsed={prn_global_elapsed:.3f}s"
        )
        _flush_partial_rows()

_flush_partial_rows()
raw_results = pd.concat([_df_prev, pd.DataFrame(new_rows)], ignore_index=True) if (_df_prev is not None and new_rows) else (_df_prev if _df_prev is not None else pd.DataFrame(new_rows))
winner_results = build_winner_table(raw_results, cfg)

export_dir = resolve_export_dir(cfg)
export_dir.mkdir(parents=True, exist_ok=True)
prefix = cfg["export_prefix"]

raw_results.to_csv(export_dir / f"{prefix}_raw_results.csv", index=False)
winner_results.to_csv(export_dir / f"{prefix}_winner_results.csv", index=False)

print("Exports:")
print(export_dir / f"{prefix}_raw_results.csv")
print(export_dir / f"{prefix}_winner_results.csv")


Acquisitions prevues: 774
Methodes: ['cpu_lut_angles']
[1/774] cpu_lut_angles PRN=25 SNR=-10 DONE global=116.313s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n10000Hz_ca-804.csv PRN=25 acquisitions=1 global_elapsed=117.741s
[2/774] cpu_lut_angles PRN=1 SNR=-10 DONE global=113.603s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n10000Hz_ca-804.csv PRN=1 acquisitions=1 global_elapsed=115.214s
[3/774] cpu_lut_angles PRN=2 SNR=-10 DONE global=113.020s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n10000Hz_ca-804.csv PRN=2 acquisitions=1 global_elapsed=114.612s
[4/774] cpu_lut_angles PRN=9 SNR=-10 DONE global=113.931s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n10000Hz_ca-804.csv PRN=9 acquisitions=1 global_elapsed=115.505s
[5/774] cpu_lut_angles PRN=11 SNR=-10 DONE global=112.984s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n10000Hz_ca-804.csv PRN=11 acquisitions=1 global_ela

[44/774] cpu_lut_angles PRN=21 SNR=-10 DONE global=109.411s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n5000Hz_ca-804.csv PRN=21 acquisitions=1 global_elapsed=110.980s
[45/774] cpu_lut_angles PRN=32 SNR=-10 DONE global=109.425s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n5000Hz_ca-804.csv PRN=32 acquisitions=1 global_elapsed=109.426s
[46/774] cpu_lut_angles PRN=25 SNR=-10 DONE global=109.403s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n6250Hz_ca-804.csv PRN=25 acquisitions=1 global_elapsed=109.404s
[47/774] cpu_lut_angles PRN=2 SNR=-10 DONE global=109.388s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n6250Hz_ca-804.csv PRN=2 acquisitions=1 global_elapsed=109.389s
[48/774] cpu_lut_angles PRN=4 SNR=-10 DONE global=109.516s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n6250Hz_ca-804.csv PRN=4 acquisitions=1 global_elapsed=109.517s
[49/774] cpu_lut_angles PRN=10 SNR=-10 

[87/774] cpu_lut_angles PRN=24 SNR=-10 DONE global=109.419s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p10000Hz_ca-804.csv PRN=24 acquisitions=1 global_elapsed=109.419s
[88/774] cpu_lut_angles PRN=28 SNR=-10 DONE global=109.412s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p10000Hz_ca-804.csv PRN=28 acquisitions=1 global_elapsed=109.413s
[89/774] cpu_lut_angles PRN=30 SNR=-10 DONE global=109.403s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p10000Hz_ca-804.csv PRN=30 acquisitions=1 global_elapsed=109.404s
[90/774] cpu_lut_angles PRN=31 SNR=-10 DONE global=109.455s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p10000Hz_ca-804.csv PRN=31 acquisitions=1 global_elapsed=109.501s
[91/774] cpu_lut_angles PRN=25 SNR=-10 DONE global=109.344s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p1250Hz_ca-804.csv PRN=25 acquisitions=1 global_elapsed=109.344s
[92/774] cpu_lut_angles PRN=1 S

[130/774] cpu_lut_angles PRN=13 SNR=-10 DONE global=109.504s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p6250Hz_ca-804.csv PRN=13 acquisitions=1 global_elapsed=109.560s
[131/774] cpu_lut_angles PRN=18 SNR=-10 DONE global=109.475s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p6250Hz_ca-804.csv PRN=18 acquisitions=1 global_elapsed=109.475s
[132/774] cpu_lut_angles PRN=22 SNR=-10 DONE global=109.486s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p6250Hz_ca-804.csv PRN=22 acquisitions=1 global_elapsed=109.487s
[133/774] cpu_lut_angles PRN=26 SNR=-10 DONE global=109.554s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p6250Hz_ca-804.csv PRN=26 acquisitions=1 global_elapsed=109.554s
[134/774] cpu_lut_angles PRN=30 SNR=-10 DONE global=109.432s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p6250Hz_ca-804.csv PRN=30 acquisitions=1 global_elapsed=109.433s
[135/774] cpu_lut_angles PRN=3

[173/774] cpu_lut_angles PRN=7 SNR=-20 DONE global=109.393s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n2500Hz_ca-804.csv PRN=7 acquisitions=1 global_elapsed=109.394s
[174/774] cpu_lut_angles PRN=9 SNR=-20 DONE global=109.404s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n2500Hz_ca-804.csv PRN=9 acquisitions=1 global_elapsed=109.405s
[175/774] cpu_lut_angles PRN=11 SNR=-20 DONE global=109.428s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n2500Hz_ca-804.csv PRN=11 acquisitions=1 global_elapsed=109.428s
[176/774] cpu_lut_angles PRN=15 SNR=-20 DONE global=109.536s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n2500Hz_ca-804.csv PRN=15 acquisitions=1 global_elapsed=109.537s
[177/774] cpu_lut_angles PRN=16 SNR=-20 DONE global=109.438s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n2500Hz_ca-804.csv PRN=16 acquisitions=1 global_elapsed=109.439s
[178/774] cpu_lut_angles PRN=21 SN

...

[561/774] cpu_lut_angles PRN=7 SNR=10 DONE global=109.398s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p1250Hz_ca-804.csv PRN=7 acquisitions=1 global_elapsed=109.398s
[562/774] cpu_lut_angles PRN=16 SNR=10 DONE global=109.444s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p1250Hz_ca-804.csv PRN=16 acquisitions=1 global_elapsed=109.445s
[563/774] cpu_lut_angles PRN=20 SNR=10 DONE global=109.404s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p1250Hz_ca-804.csv PRN=20 acquisitions=1 global_elapsed=109.404s
[564/774] cpu_lut_angles PRN=27 SNR=10 DONE global=109.463s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p1250Hz_ca-804.csv PRN=27 acquisitions=1 global_elapsed=109.463s
[565/774] cpu_lut_angles PRN=28 SNR=10 DONE global=109.366s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p1250Hz_ca-804.csv PRN=28 acquisitions=1 global_elapsed=109.366s
[566/774] cpu_lut_angles PRN=31 SNR=1

[604/774] cpu_lut_angles PRN=25 SNR=10 DONE global=109.454s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p7500Hz_ca-804.csv PRN=25 acquisitions=1 global_elapsed=109.454s
[605/774] cpu_lut_angles PRN=7 SNR=10 DONE global=109.467s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p7500Hz_ca-804.csv PRN=7 acquisitions=1 global_elapsed=109.468s
[606/774] cpu_lut_angles PRN=14 SNR=10 DONE global=109.462s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p7500Hz_ca-804.csv PRN=14 acquisitions=1 global_elapsed=109.462s
[607/774] cpu_lut_angles PRN=15 SNR=10 DONE global=109.433s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p7500Hz_ca-804.csv PRN=15 acquisitions=1 global_elapsed=109.433s
[608/774] cpu_lut_angles PRN=16 SNR=10 DONE global=109.429s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p7500Hz_ca-804.csv PRN=16 acquisitions=1 global_elapsed=109.430s
[609/774] cpu_lut_angles PRN=17 SNR=1

[647/774] cpu_lut_angles PRN=28 SNR=20 DONE global=109.581s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n2500Hz_ca-804.csv PRN=28 acquisitions=1 global_elapsed=109.581s
[648/774] cpu_lut_angles PRN=31 SNR=20 DONE global=109.505s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n2500Hz_ca-804.csv PRN=31 acquisitions=1 global_elapsed=109.506s
[649/774] cpu_lut_angles PRN=25 SNR=20 DONE global=109.612s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n3750Hz_ca-804.csv PRN=25 acquisitions=1 global_elapsed=109.613s
[650/774] cpu_lut_angles PRN=7 SNR=20 DONE global=109.525s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n3750Hz_ca-804.csv PRN=7 acquisitions=1 global_elapsed=109.696s
[651/774] cpu_lut_angles PRN=8 SNR=20 DONE global=109.618s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n3750Hz_ca-804.csv PRN=8 acquisitions=1 global_elapsed=109.619s
[652/774] cpu_lut_angles PRN=15 SNR=20 

[690/774] cpu_lut_angles PRN=19 SNR=20 DONE global=109.519s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n8750Hz_ca-804.csv PRN=19 acquisitions=1 global_elapsed=109.698s
[691/774] cpu_lut_angles PRN=20 SNR=20 DONE global=109.511s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n8750Hz_ca-804.csv PRN=20 acquisitions=1 global_elapsed=109.511s
[692/774] cpu_lut_angles PRN=24 SNR=20 DONE global=109.558s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n8750Hz_ca-804.csv PRN=24 acquisitions=1 global_elapsed=109.559s
[693/774] cpu_lut_angles PRN=30 SNR=20 DONE global=109.482s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n8750Hz_ca-804.csv PRN=30 acquisitions=1 global_elapsed=109.483s
[694/774] cpu_lut_angles PRN=25 SNR=20 DONE global=109.496s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p0Hz_ca-804.csv PRN=25 acquisitions=1 global_elapsed=109.496s
[695/774] cpu_lut_angles PRN=2 SNR=20 

[733/774] cpu_lut_angles PRN=20 SNR=20 DONE global=109.516s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p3750Hz_ca-804.csv PRN=20 acquisitions=1 global_elapsed=109.517s
[734/774] cpu_lut_angles PRN=22 SNR=20 DONE global=109.492s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p3750Hz_ca-804.csv PRN=22 acquisitions=1 global_elapsed=109.493s
[735/774] cpu_lut_angles PRN=24 SNR=20 DONE global=109.503s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p3750Hz_ca-804.csv PRN=24 acquisitions=1 global_elapsed=109.504s
[736/774] cpu_lut_angles PRN=27 SNR=20 DONE global=109.582s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p3750Hz_ca-804.csv PRN=27 acquisitions=1 global_elapsed=109.583s
[737/774] cpu_lut_angles PRN=28 SNR=20 DONE global=109.607s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p3750Hz_ca-804.csv PRN=28 acquisitions=1 global_elapsed=109.607s
[738/774] cpu_lut_angles PRN=30 SNR